# toolUsage — from raw transcripts to the dev-card number

This notebook walks the **exact production pipeline** that produces this
metric's value on the profile page, starting from raw `~/.claude/projects/**/*.jsonl`
transcripts. Every step prints an interim value so you can see what the extractor
is doing.

**Formula (LaTeX):**

```
(\text{builtin\_tool\_invocations},\ \text{distinct\_builtin\_tools})
```

**Parts:**

- **builtin_tool_invocations** = `builtin_tool_invocations` (calls) — Total invocations of built-in Claude Code tools (Bash, Read, Edit, Write, Grep, Glob, Task, TodoWrite, WebFetch, WebSearch, etc.) across the window — i.e. tools that aren't MCP, skill, or plugin commands.
- **distinct_builtin_tools** = `distinct_builtin_tools` (tools) — Number of distinct built-in tool names invoked at least once across the window.

**Server aggregator:** `server/web/lib/scorer/metrics/toolUsage.ts`
**Wire fields read:** `sessions[].builtin_tool_invocations`, `sessions[].distinct_builtin_tools`


## Step 1 — Discover sessions

List every JSONL transcript under `~/.claude/projects/`.


In [1]:
from pathlib import Path
import sys, os, json, uuid
# Make the `scripts/` package importable regardless of where the
# notebook is opened from. Walk upward from cwd until we find a
# directory containing `scripts/extractor.py` (the client root).
_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / 'scripts' / 'extractor.py').exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
    if _p.parent == _p:
        break
    _p = _p.parent
else:
    raise RuntimeError(
        'could not locate the client root (looking for scripts/extractor.py). '
        f'Started from {Path.cwd()}.'
    )
from scripts.events import find_sessions
from scripts.session_labels import build_session_labels

sessions = find_sessions()
# Map session_id and session_hash -> 'aiTitle' (Claude Code's own
# session title) with a quoted first-user-words fallback. Used
# below wherever a session is identified so you can grep the
# archive for the label, not just the hex id.
session_labels = build_session_labels(sessions)
print(f'discovered {len(sessions)} JSONL transcripts')
for s in sessions[:5]:
    label = session_labels.get(s.session_id, '')
    print(f'  {s.session_id[:12]}  {label}'.rstrip())
    print(f'    project={s.project_root}  first={s.first_ts_ms}  last={s.last_ts_ms}')


discovered 80 JSONL transcripts
  b9da9f91-5d5  Troubleshoot Claude cowork sandbox access
    project=/home/alonb/aiqrank  first=1778087890660  last=1778087997962
  65d5bf8a-b6c  Review codebase architecture and functionality
    project=/home/alonb/aiqrank  first=1777491237360  last=1777495378001
  fc883668-4cc  Find GitHub repository
    project=/home/alonb/aiqrank  first=1779549935629  last=1779549940333
  2de58a54-f17  List repository files with GitHub MCP
    project=/home/alonb/agentscore  first=1779537407300  last=1779537599012
  6ab4bcbb-e1d  Install official GitHub MCP server
    project=/home/alonb/agentscore  first=1779547138200  last=1779547582912


## Step 2 — Anchor to the last upload, then run the extractor

ConductorScore's Supabase schema does **not** preserve the wire payload —
only the per-metric `raw` values the server computed (in the `scores`
table). That means we can't replay the original payload byte-for-byte,
but we can anchor the 30-day window to the upload's `extracted_at_ms` so
the local re-extract sees the same sessions the upload did.

Two modes — pick via `MODE` below:

- `"reextract_anchored"` (default when Supabase creds are present) — pulls
  the upload's anchor timestamp from `devices.last_upload_at` and feeds it
  to `extract(now_ms=…)`. Window matches exactly; values may still differ
  from the live UI if the extractor logic changed since the upload (e.g.
  the `Agent` tool-name fix this branch shipped, or wire-schema bumps).
- `"reextract_now"` — anchors to current time. Drifts by whatever has
  happened since the upload. Useful for "what would my score be today?"


In [2]:
import time
from scripts.extractor import extract
from scripts.debug_metric.collectors.uploads_body import fetch_last_upload, UploadNotFound

USERNAME = 'jswift24'         # change to your GitHub handle
MODE = 'reextract_anchored'   # 'reextract_anchored' | 'reextract_now'

upload = None
try:
    upload = fetch_last_upload(USERNAME)
    from datetime import datetime, timezone
    dt = datetime.fromtimestamp(upload['extracted_at_ms'] / 1000, tz=timezone.utc)
    print(f'last upload: {upload["hours_ago"]}h ago, extracted_at_ms={upload["extracted_at_ms"]} ({dt.isoformat()})')
    print(f'live score row: composite={upload["composite"]:.3f} tier={upload["tier"]} computed_at={upload["computed_at"]}')
except (UploadNotFound, RuntimeError) as e:
    print(f'⚠ could not fetch last upload ({e}); falling back to reextract_now')
    MODE = 'reextract_now'

if MODE == 'reextract_anchored' and upload is not None:
    now_ms = upload['extracted_at_ms']
    label = f'anchored to upload ({now_ms})'
else:
    now_ms = int(time.time() * 1000)
    label = f'anchored to now ({now_ms})'

payload = extract(device_id=str(uuid.UUID(int=0)), client_version='notebook', now_ms=now_ms)
wire = json.loads(payload.to_json())
print(f'extracted {len(wire["sessions"])} sessions {label}')
print(f'schema_version: {wire["device"]["schema_version"]}')


⚠ could not fetch last upload (SUPABASE_URL is not set. Export it before fetching the last upload.); falling back to reextract_now


extracted 79 sessions anchored to now (1779848168643)
schema_version: 0.8


## Step 3 — Inspect the wire fields this metric reads

Per the registry, this metric reads the following wire fields:

- `sessions[].builtin_tool_invocations`
- `sessions[].distinct_builtin_tools`

Below are the per-session values for those fields, plus a small distribution summary.


In [3]:
WIRE_FIELDS = ["sessions[].builtin_tool_invocations", "sessions[].distinct_builtin_tools"]

def read(path: str, session: dict):
    if path.startswith('sessions[].'):
        return session.get(path[len('sessions[].'):])
    if path.startswith('config.'):
        return wire.get('config', {}).get(path[len('config.'):])
    return wire.get(path)

for field in WIRE_FIELDS:
    if field.startswith('config.'):
        print(f'{field}: {read(field, {})!r}')
        continue
    values = [read(field, s) for s in wire['sessions']]
    nonzero = [v for v in values if v not in (0, None, [], False)]
    print(f'{field}: {len(nonzero)} non-zero sessions, top 5: ' + str(
        sorted([v for v in nonzero if isinstance(v, (int, float))], reverse=True)[:5] or nonzero[:5]
    ))


sessions[].builtin_tool_invocations: 79 non-zero sessions, top 5: [474, 389, 324, 299, 294]
sessions[].distinct_builtin_tools: 79 non-zero sessions, top 5: [['Agent', 'Bash'], ['Bash', 'Read'], ['Bash'], ['Agent', 'Bash', 'Read'], ['Bash', 'Edit', 'Read', 'Write']]


## Step 4 — Apply the server aggregator

The TypeScript aggregator at `server/web/lib/scorer/metrics/toolUsage.ts` consumes the
wire payload via the same `aggregate-one.ts` adapter the debug script uses.
We shell out to it through `npx tsx` and parse the result. The output is
the canonical `{ raw, parts }` shape that the API surfaces and the dev-card
tile renders.


In [4]:
import subprocess
from scripts.debug_metric.registry import _SERVER_WEB, _node20_path_env

result = subprocess.run(
    ['npx', 'tsx', 'scripts/aggregate-one.ts', 'toolUsage'],
    cwd=_SERVER_WEB,
    input=json.dumps(wire),
    capture_output=True, text=True, check=True,
    env=_node20_path_env(),
)
agg = json.loads(result.stdout)
print(json.dumps(agg, indent=2))


{
  "raw": {
    "invocations": 4579,
    "distinct": 16
  },
  "sign": "informational",
  "parts": [
    {
      "symbol": "I",
      "value": 4579,
      "unit": "calls"
    },
    {
      "symbol": "D",
      "value": 16,
      "unit": "tools"
    }
  ]
}


## Step 5 — Format like the dev-card

The profile tile renders the `raw` field with metric-specific formatting.
This cell mirrors `components/dev-card.tsx` exactly for `toolUsage` — the
final printed string matches what `/u/<your-handle>` displays for the
most recent upload.


In [5]:
raw = agg.get('raw')
parts = agg.get('parts', [])

# Per-metric formatters mirror components/dev-card.tsx exactly.
# Each entry produces (headline, suffix) like the prototype's
# `<span class="val">{headline}<small>{suffix}</small></span>`.
def fmt_dev_card(metric_id, raw):
    if raw is None:
        return '—', ''
    if metric_id == 'humanAgentWallclock':
        # raw = {human: minutes, agent: minutes}; UI shows 'H / A h'.
        h = round(raw['human'] / 60); a = round(raw['agent'] / 60)
        return f'{h} / {a}', ' h'
    if metric_id == 'agentParallelism':
        return f'{raw:.2f}', '×'
    if metric_id == 'agentMaxRuntime':
        return f'{raw}', ' min'
    if metric_id == 'codingWithoutPlan':
        return f'{round(raw * 100)}', '% of significant edits'
    if metric_id == 'autoCompactionRate':
        return f'{raw:.1f}', ' per 100k tokens'
    if metric_id == 'revertRate':
        return f'{raw:.1f}', ' per coding session'
    if metric_id == 'repetitivePrompts':
        return f'{round(raw * 100)}', '% of long prompts'
    if metric_id == 'claudeMdBloat':
        return f'{raw}', ' lines'
    if metric_id == 'redundantApprovals':
        return f'{raw:.1f}', ' per session'
    if metric_id == 'rageQuit':
        return f'{round(raw * 100)}', ' per 100 sessions'
    if metric_id == 'topTierShare':
        # Scorer stores raw = 1 - max_tier_share. UI shows (1-raw)*100.
        return f'{round((1 - raw) * 100)}', '% on most expensive model'
    if metric_id == 'modelFreshness':
        return f'{round(raw * 100)}', '% on current ver.'
    if metric_id in ('mcpUsage', 'skillUsage', 'toolUsage', 'pluginUsage'):
        return f"{raw['invocations']} / {raw['distinct']}", ''
    if metric_id == 'costAggregate':
        if raw < 1: return '<$1', ''
        return f'${round(raw):,}', ''
    if metric_id == 'tokensAggregate':
        total = raw['total'] if isinstance(raw, dict) else raw
        if total >= 1_000_000: return f'{round(total / 1_000_000):,}M', ''
        if total >= 1_000:     return f'{round(total / 1_000):,}k', ''
        return f'{total:,}', ''
    return f'{raw}', ''

headline, suffix = fmt_dev_card('toolUsage', raw)
print(f'\n=== DEV-CARD VALUE ===')
print(f'{headline}{suffix}')
print(f'======================')
print()
print('Sub-metrics (shown in the (i) modal):')
for p in parts:
    print(f'  {p["symbol"]} = {p["value"]:,} {p["unit"]}')



=== DEV-CARD VALUE ===
4579 / 16

Sub-metrics (shown in the (i) modal):
  I = 4,579 calls
  D = 16 tools


## Step 6 — Compare to the live `scores` row

Supabase's `scores.metrics.<id>.raw` is the value the dev card
renders. Below we read it and diff against what Step 4 computed
locally. Any divergence means the extractor / aggregator logic
moved between the upload and now (e.g. the `Agent` tool-name fix,
or a v0.2+ wire-schema bump that this metric reads).


In [6]:
current_id = 'toolUsage'
legacy_id = None
# Metrics that require server-side context (Supabase model_pricing,
# endoflife.date freshness) — local re-extract via aggregate-one.ts
# uses empty defaults, so a 0 / <\$1 local result is expected, not buggy.
DEGRADED_METRICS = {'costAggregate', 'modelFreshness'}

live_raw = None
live_source = None
if upload is not None:
    scored = upload['scored_raws']
    if current_id in scored:
        live_raw, live_source = scored[current_id], current_id
    elif legacy_id and legacy_id in scored:
        live_raw, live_source = scored[legacy_id], f'{legacy_id} (legacy)'

if live_raw is None:
    print('— no live scored value available (upload predates this metric, or no Supabase creds)')
else:
    local_raw = agg.get('raw')
    print(f'live  {live_source!r:36} raw={live_raw!r}')
    print(f'local {current_id!r:36} raw={local_raw!r}')

    def _close(a, b, rel_tol=1e-2, abs_tol=1e-4):
        # Numeric closeness with both absolute and relative tolerance.
        # rel_tol=1% catches the JSONL-growth drift (typically <1% over
        # a few minutes of additional usage); abs_tol=1e-4 handles small-
        # value metrics like rates near zero.
        if isinstance(a, dict) and isinstance(b, dict):
            return set(a) == set(b) and all(_close(a[k], b[k], rel_tol, abs_tol) for k in a)
        if isinstance(a, list) and isinstance(b, list):
            return len(a) == len(b) and all(_close(x, y, rel_tol, abs_tol) for x, y in zip(a, b))
        try:
            af, bf = float(a), float(b)
        except (TypeError, ValueError):
            return a == b
        return abs(af - bf) <= max(abs_tol, rel_tol * max(abs(af), abs(bf)))

    def _delta_summary(a, b):
        # Human-readable summary of the gap between live and local.
        if isinstance(a, dict) and isinstance(b, dict):
            parts = []
            for k in sorted(set(a) | set(b)):
                if k not in a or k not in b: parts.append(f'{k}: missing')
                elif a[k] != b[k]:
                    try:
                        d = float(b[k]) - float(a[k])
                        parts.append(f'{k}: {d:+g}')
                    except Exception:
                        parts.append(f'{k}: differs')
            return '; '.join(parts) if parts else 'equal'
        try:
            d = float(b) - float(a)
            return f'{d:+g} ({d/max(abs(float(a)),1e-9)*100:+.2f}%)'
        except Exception:
            return f'{a!r} vs {b!r}'

    if _close(live_raw, local_raw):
        print('\n✓ MATCH — local value matches the live UI within tolerance (1% rel / 1e-4 abs).')
    elif current_id in DEGRADED_METRICS:
        print(f'\nⓘ DEGRADED — local raw is the documented `<\$1` / 0 fallback because',
              f'aggregate-one.ts has no Supabase context (pricing for cost,',
              f'endoflife.date for freshness). The live UI runs against real data.')
    else:
        delta = _delta_summary(live_raw, local_raw)
        print(f'\n⚠ DIVERGENCE — delta: {delta}')
        print(f'  Most-likely cause: JSONL growth — you continued using Claude Code')
        print(f"  in the {upload['hours_ago']}h since the upload, adding events that")
        print(f'  this re-extract sees but the upload did not. To bridge: re-run')
        print(f'  /conductorscore right before re-running this notebook.')


— no live scored value available (upload predates this metric, or no Supabase creds)


<>:61: SyntaxWarning: invalid escape sequence '\$'
<>:61: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_89732/1356100170.py:61: SyntaxWarning: invalid escape sequence '\$'
  print(f'\nⓘ DEGRADED — local raw is the documented `<\$1` / 0 fallback because',


## Where this lands on the page

The number printed by Step 5 is what the dev-card tile renders for `toolUsage`.
Open `https://conductorscore.com/u/<your-handle>` and find the tile matching
selector `[data-tile="tool-usage"]`. Click the **ⓘ** button to see the same parts
breakdown printed above, with each symbol annotated by its `label` and `describe`.

For end-to-end provenance (UI ↔ API ↔ DB ↔ upload ↔ this re-extract), run:

```bash
python -m scripts.debug_metric toolUsage --user <your-handle>
```
